In [2]:
import pymupdf
import re

from pathlib import Path

# Create project root
def find_project_root(marker: str = '.git') -> Path:
    '''Climbs up folders starting from CWD until it finds the root.'''
    current = Path.cwd().resolve()

    # Loop upward until hitting the project root (where current == its own parent)
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent

PROJECT_ROOT = find_project_root(marker='.git')
PDF_DIR = PROJECT_ROOT / 'data' / 'raw' / 'records'

## Analyze Project Descriptions

Standard Permit documents differs from PSD/GHGPSD documents. Furthermore the document structures differ among the same permit types depending on the time period so the entire first page is passed to the agent. 

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

async def analyze_proj_desc(cover_text: str) -> dict:
    schema = {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "site_name": {"type": "string"},
                "county": {"type": "string"},
                "is_data_center": {"type": "boolean"},
                "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
                "capacity_MW": {"type": "number"},
            },
            "required": ["site_name", "county", "is_data_center", "confidence", "capacity_MW"]
        }
    }

    stderr_lines = []

    options = ClaudeAgentOptions(
        system_prompt = '''
            You are a document classifier for TCEQ air permit filings. Given a project description, determine whether it describes a gas-fired power generation facility built to power an onsite or immediately-adjacent data center / compute facility behind the meter (i.e., generation dedicated to the data center's load, not routed through the grid).

            Does NOT qualify:
            - Crypto-mining facilities (regardless of power source)
            - Grid-interconnected generation, even if colocated with other industrial loads
            - Generation whose stated purpose is oilfield/production support (compressors, artificial lift, etc.) rather than compute load
            - Generation serving mixed/unspecified loads with no data center mentioned

            Only treat it as ambiguous, and only then use web search, if the project description names a facility/operator but does not state its purpose clearly. Do at most one round of search; if search doesn't resolve it, classify at "low" confidence rather than continuing to search.

            Confidence rubric:
            - "high": the document explicitly states the generation powers a data center/compute facility
            - "medium": the document implies it (e.g., site name, operator, or context strongly suggests it) but doesn't state it outright
            - "low": classification relies on external web search or is a close judgment call

            If the facility qualifies, extract operating capacity in MW from the document; use -1 if not stated.
        ''',
        max_turns=25,
        model="claude-sonnet-5",
        output_format=schema,
        stderr=stderr_lines.append,
    )

    prompt = f"Project Description: {cover_text}"

    messages = []
    async for message in query(prompt=prompt, options=options):
        messages.append(message)
        if isinstance(message, ResultMessage):
            if message.is_error:
                raise RuntimeError(f"Result error ({message.subtype}): {message.errors}")
            if message.structured_output is not None:
                return message.structured_output
            raise RuntimeError(f"ResultMessage had no structured_output (subtype={message.subtype})")

    print("No ResultMessage received. Full message sequence:")
    for m in messages:
        print(" -", m)
    if stderr_lines:
        print("CLI stderr:")
        for line in stderr_lines:
            print(" ", line)
    raise RuntimeError("No result message received")

In [59]:
pdf_paths = list(PDF_DIR.glob("*.pdf"))

def extract_section(prefix: str, suffix: str, text):
    pattern = rf"{prefix}(.*?){suffix}"

    return re.search(pattern, text, re.DOTALL).group(1)

proj_infos = []
for path in pdf_paths:
    cover_text = pymupdf.open(path)[0].get_text()

    proj_info = await analyze_proj_desc(cover_text)
    proj_infos.append(proj_info)


RuntimeError: No result message received

In [57]:
proj_infos

[None, None, None, None, None, None, None, None]

In [47]:
import polars as pl

site_records = pl.DataFrame(
    {
        key: [
            data[key]
            for data in proj_infos
        ]
        for key in proj_infos[0].keys()
    },
    strict=False
)

data_centers = site_records.filter(
    pl.col("is_data_center") == True
)

data_centers

site_name,county,is_data_center,confidence,capacity_MW
str,str,bool,str,f64
"""Circe Energy Data Centers E Mo…","""Ward""",true,"""high""",400.0
"""GW Ranch Energy Center""","""Pecos""",true,"""high""",5000.0
"""Rock House Draw Generating Sta…","""Pecos""",true,"""high""",576.0
"""Pecos Power Generation""","""Reeves""",true,"""high""",478.0
"""Xenergy Monahans 1""","""Ward""",true,"""high""",118.98
"""Featherwood Energies Pecos Pla…","""Reeves""",true,"""high""",931.0
"""Poolside Data Center Pecos Cou…","""Pecos""",true,"""high""",332.28


In [ ]:
async def extract_engines(doc_text: str) -> str:
    return doc_text



In [63]:
EX_PATH = PDF_DIR / 'AIR NSR_174494-410474_Permits_Public_20260604_Technical Review_8477114_.pdf'

doc_text = pymupdf.open(EX_PATH)[0].get_text()

def extract_section(prefix: str, suffix: str, text):
    pattern = rf"{prefix}(.*?){suffix}"

    return re.search(pattern, text, re.DOTALL).group(1)

proj_desc = extract_section("Project Description", "Compliance", doc_text)
county_name = extract_section("County", "Regulated", doc_text)
site_name = extract_section("Site Name", "Project Description", doc_text)


'E'